# 03 · Two variables, two questions

## Context

A field campaign has collected temperature and precipitation on the same grid.
The variables align in space and time, but they answer different questions and
carry different units.

## Question

Where is the latest temperature unusual, and how did average precipitation
change across the region?

## Analysis story

We will keep both variables in one xarray `Dataset`, select each by meaning, and
write a separate compact pipe for each scientific question.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from cubedynamics import pipe, verbs as v

# A fixed seed makes the example exactly repeatable while still looking like
# measurements with natural variation.
rng = np.random.default_rng(19)
time = pd.date_range("2024-04-01", periods=20, freq="D")
y = np.linspace(41.0, 40.0, 4)
x = np.linspace(-106.0, -105.0, 5)
temperature = 18 + np.linspace(0, 7, time.size)[:, None, None] + rng.normal(0, 1, (20, 4, 5))
precipitation = rng.gamma(1.4, 2.2, size=(20, 4, 5))

# A Dataset is a labeled collection of aligned DataArrays. Each variable has
# its own units but shares the same time/y/x coordinate system.
dataset = xr.Dataset(
    {
        "temperature": (("time", "y", "x"), temperature, {"units": "degC"}),
        "precipitation": (("time", "y", "x"), precipitation, {"units": "mm day-1"}),
    },
    coords={"time": time, "y": y, "x": x},
    attrs={"source": "deterministic multi-variable example"},
)
dataset

## Pipes · Let each question choose its verb

The first pipe preserves a cube of anomalies. The second deliberately reduces
the two spatial dimensions to one regional time series.

In [ ]:
# Question 1: how unusual is temperature at every grid cell?
temperature_anomaly = (
    pipe(dataset["temperature"])
    | v.anomaly(dim="time")
).unwrap()

# Question 2: what was the region-wide precipitation on each date?
regional_precipitation = (
    pipe(dataset["precipitation"])
    | v.mean(dim=("y", "x"), keep_dim=False)
).unwrap()

assert temperature_anomaly.dims == ("time", "y", "x")
assert regional_precipitation.dims == ("time",)

## Figure · Bring the answers together

The outputs have different shapes because the questions differ: a map for
spatial departures and a line for regional change through time.

In [ ]:
import matplotlib.pyplot as plt

# The side-by-side views answer different questions from the same Dataset.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), constrained_layout=True)
temperature_anomaly.isel(time=-1).plot(
    ax=axes[0], cmap="RdBu_r", center=0, cbar_kwargs={"label": "°C anomaly"}
)
axes[0].set_title("Latest temperature anomaly")
regional_precipitation.plot(ax=axes[1], marker="o", color="#2f6267")
axes[1].set_title("Spatial mean precipitation")
axes[1].set_ylabel("mm day⁻¹")
plt.show()

## What the figure tells us

Temperature departures vary across the final map, while precipitation is
summarized as one value per date. A shared `Dataset` does not force a shared
analysis; each short pipe makes its own question and output shape explicit.

## Try the next variation

Compute a precipitation anomaly instead of a spatial mean. How does the output
shape—and therefore the figure you would choose—change?